In [3]:
from helpers import load_csv_data 
import numpy as np
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("dataset")
print(x_train.shape, x_test.shape)
print(np.unique(y_train, return_counts=True))
print(np.isnan(x_train).mean(axis=0)) 

KeyboardInterrupt: 

In [ ]:
with open("dataset/x_train.csv") as f:
    names = np.array(f.readline().strip().split(",")[1:])

In [ ]:
x_train_raw, x_test_raw, names_raw = x_train, x_test, names 

Only keep the ones that have less than 70% missing.

In [ ]:
missing = np.isnan(x_train_raw).mean(axis=0)
std = np.nanstd(x_train, axis=0)
keep = (missing < 0.8)
x_train, x_test, names = x_train_raw[:, keep], x_test_raw[:, keep], names_raw[keep]
print(len(names))

205


In [2]:
"""Baseline pipeline for Project 1.

Steps:
  1. Load the data (cached as .npz after the first, slow CSV load).
  2. Clean: recode special survey answers, drop bookkeeping columns,
     drop mostly-missing and constant columns.
  3. Compare methods from class with 5-fold cross-validation (F1 score),
     choosing the decision threshold on the out-of-fold predictions.
  4. Retrain the best method on all training data and write a submission.
"""

import os
import time

import numpy as np

from helpers import load_csv_data, create_csv_submission
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    ridge_regression,
)

DATA_DIR = "dataset"
SEED = 42
K_FOLDS = 5
MAX_MISSING = 0.8  # drop columns missing for more than 30% of people
MAX_ITERS = 1000
GAMMA = 0.1

# Columns describing the phone interview itself, not the person.
ADMIN_COLUMNS = [
    "_STATE", "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE",
    "SEQNO", "_PSU", "CTELENUM", "PVTRESD1", "COLGHOUS", "STATERES",
    "CELLFON3", "LADULT", "NUMADULT", "NUMMEN", "NUMWOMEN", "CTELNUM1",
    "CELLFON2", "CADULT", "PVTRESD2", "CCLGHOUS", "CSTATE", "LANDLINE",
    "HHADULT",
]

# Survey codes meaning "don't know" / "refused" -> treated as missing.
MISSING_CODES = {
    "GENHLTH": [7, 9],
    "PHYSHLTH": [77, 99],
    "MENTHLTH": [77, 99],
    "POORHLTH": [77, 99],
    "CHILDREN": [99],
    "INCOME2": [77, 99],
    "WEIGHT2": [7777, 9999],
    "HEIGHT3": [7777, 9999],
    "ALCDAY5": [777, 999],
    "AVEDRNK2": [77, 99],
    "DRNK3GE5": [77, 99],
    "MAXDRNKS": [77, 99],
    "FRUITJU1": [777, 999],
    "FRUIT1": [777, 999],
    "FVBEANS": [777, 999],
    "FVGREEN": [777, 999],
    "FVORANG": [777, 999],
    "VEGETAB1": [777, 999],
    "EXEROFT1": [777, 999],
    "EXERHMM1": [777, 999],
    "EXEROFT2": [777, 999],
    "EXERHMM2": [777, 999],
    "STRENGTH": [777, 999],
    "JOINPAIN": [77, 99],
    "FLSHTMY2": [777777, 999999],
    "HIVTSTD3": [777777, 999999],
}
# Survey codes meaning "none" / "never" -> replaced by 0.
NONE_CODES = {
    "PHYSHLTH": [88],
    "MENTHLTH": [88],
    "POORHLTH": [88],
    "CHILDREN": [88],
    "ALCDAY5": [888],
    "DRNK3GE5": [88],
    "FRUITJU1": [555],
    "FRUIT1": [555],
    "FVBEANS": [555],
    "FVGREEN": [555],
    "FVORANG": [555],
    "VEGETAB1": [555],
    "STRENGTH": [888],
}


# ---------------------------------------------------------------- data


def load_data():
    """Load train/test data and column names, using a cache if available."""
    cache = os.path.join(DATA_DIR, "cache.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        x_train, x_test, y_train = d["x_train"], d["x_test"], d["y_train"]
        train_ids, test_ids = d["train_ids"], d["test_ids"]
    else:
        print("Loading CSV files (slow the first time)...")
        x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_DIR)
        np.savez(
            cache, x_train=x_train, x_test=x_test, y_train=y_train,
            train_ids=train_ids, test_ids=test_ids,
        )
    with open(os.path.join(DATA_DIR, "x_train.csv")) as f:
        names = np.array(f.readline().strip().split(",")[1:])  # skip Id
    return x_train, x_test, y_train, test_ids, names


# ------------------------------------------------------- preprocessing


def recode_special_values(x, names):
    """Replace survey special codes with NaN (unknown) or 0 (none)."""
    x = x.copy()
    for name, codes in MISSING_CODES.items():
        for j in np.flatnonzero(names == name):
            x[np.isin(x[:, j], codes), j] = np.nan
    for name, codes in NONE_CODES.items():
        for j in np.flatnonzero(names == name):
            x[np.isin(x[:, j], codes), j] = 0
    return x


def select_columns(x_train, x_test, names):
    """Recode special values, then drop bookkeeping, mostly-missing and
    constant columns. Uses no labels, so it is done once on the full
    training set (this keeps memory use low during cross-validation)."""
    x_train = recode_special_values(x_train, names)
    x_test = recode_special_values(x_test, names)
    missing = np.isnan(x_train).mean(axis=0)
    std = np.nanstd(x_train, axis=0)
    keep = (missing < MAX_MISSING) & (std > 0) & ~np.isin(names, ADMIN_COLUMNS + NOMINAL_COLUMNS)
    return x_train[:, keep], x_test[:, keep], names[keep]

MAX_CATEGORIES = 10


def expand_features(x_train, x_test, names):
    """One-hot encode categorical columns and add missing-value indicators
    for continuous ones. Categories are learned on the training set only;
    a category unseen in training gives all zeros."""
    specs = []
    for j, name in enumerate(names):
        col = x_train[:, j]
        values = np.unique(col[~np.isnan(col)])
        if len(values) <= MAX_CATEGORIES:
            for v in values:
                specs.append(("category", j, v, f"{name}={v:g}"))
        else:
            specs.append(("value", j, None, name))
            if np.isnan(col).any():
                specs.append(("missing", j, None, f"{name}_missing"))

    def build(x):
        out = np.empty((len(x), len(specs)), dtype=np.float32)
        for k, (kind, j, v, _) in enumerate(specs):
            if kind == "category":
                out[:, k] = x[:, j] == v
            elif kind == "missing":
                out[:, k] = np.isnan(x[:, j])
            else:
                out[:, k] = x[:, j]
        return out

    new_names = np.array([s[3] for s in specs])
    return build(x_train), build(x_test), new_names

def fit_scaler(x):
    """Learn imputation medians and standardization stats on training data."""
    medians = np.nanmedian(x, axis=0)
    filled = np.where(np.isnan(x), medians, x)
    mean, std = filled.mean(axis=0), filled.std(axis=0)
    std[std == 0] = 1
    return medians, mean, std


def transform(x, scaler):
    """Impute, standardize and add a bias column, using training stats."""
    medians, mean, std = scaler
    x = np.where(np.isnan(x), medians, x)
    x = (x - mean) / std
    return np.c_[np.ones(len(x), dtype=np.float32), x]


# -------------------------------------------------------------- models
# Each returns weights w; predictions are scores tx @ w compared to a
# threshold (for logistic regression this is equivalent to thresholding
# the probability, since the sigmoid is increasing).

MODELS = {
    "ridge regression (lambda=1e-5)": lambda y, tx: ridge_regression(
        y, tx, 1e-5
    )[0],
    "logistic regression": lambda y, tx: logistic_regression(
        y, tx, np.zeros(tx.shape[1], dtype=np.float32), MAX_ITERS, GAMMA
    )[0],
    "reg. logistic regression (lambda=1e-3)": lambda y, tx: reg_logistic_regression(
        y, tx, 1e-3, np.zeros(tx.shape[1],dtype=np.float32), MAX_ITERS, GAMMA
    )[0],
}


# ---------------------------------------------------------- evaluation


def f1_score(y_true, y_pred):
    """F1 score for y_true in {0, 1} and boolean predictions."""
    positive = y_true == 1
    tp = np.sum(y_pred & positive)
    fp = np.sum(y_pred & ~positive)
    fn = np.sum(~y_pred & positive)
    return 2 * tp / (2 * tp + fp + fn) if tp > 0 else 0.0


def best_threshold(y, scores):
    """Threshold on scores that maximizes F1 (tries predicting the top
    1% to 50% of scores as positive)."""
    candidates = np.quantile(scores, np.linspace(0.5, 0.99, 50))
    f1s = [f1_score(y, scores >= t) for t in candidates]
    i = int(np.argmax(f1s))
    return candidates[i], f1s[i]


def cross_validate(train_fn, x, y):
    """K-fold CV: out-of-fold scores, then the best threshold and its F1."""
    rng = np.random.default_rng(SEED)
    folds = np.array_split(rng.permutation(len(y)), K_FOLDS)
    scores = np.empty(len(y))
    for k in range(K_FOLDS):
        va = folds[k]
        tr = np.concatenate([folds[i] for i in range(K_FOLDS) if i != k])
        scaler = fit_scaler(x[tr])
        w = train_fn(y[tr], transform(x[tr], scaler))
        scores[va] = transform(x[va], scaler) @ w
        print(f"  fold {k + 1}/{K_FOLDS} done")
    threshold, f1 = best_threshold(y, scores)
    predicted = scores >= threshold                # True = predicted positive
    accuracy = np.mean(predicted == (y == 1))     # fraction of matches
    return threshold, f1, accuracy


# ---------------------------------------------------------------- main


def main():
    x_train, x_test, y_train, test_ids, names = load_data()
    y = (y_train == 1).astype(np.float32)  # labels {-1, 1} -> {0, 1}

    x_train, x_test, names = select_columns(x_train, x_test, names)
    x_train, x_test, names = expand_features(x_train, x_test, names)
    print(f"Using {len(names)} features, {len(y)} training samples")

    results = {}
    for model_name, train_fn in MODELS.items():
        start = time.time()
        threshold, f1, accuracy = cross_validate(train_fn, x_train, y)
        results[model_name] = (f1, threshold)
        print(f"{model_name:42s} CV F1 = {f1:.4f}  accuracy = {accuracy:.4f}  "f"({time.time() - start:.0f}s)")

    best = max(results, key=lambda name: results[name][0])
    f1, threshold = results[best]
    print(f"Best: {best} (CV F1 = {f1:.4f})")

    # Retrain on all training data and predict the test set.
    scaler = fit_scaler(x_train)
    w = MODELS[best](y, transform(x_train, scaler))
    test_scores = transform(x_test, scaler) @ w
    predictions = np.where(test_scores >= threshold, 1, -1)

    os.makedirs("submissions", exist_ok=True)
    create_csv_submission(test_ids, predictions, "submissions/submission.csv")
    print(f"Saved submissions/submission.csv "
          f"({np.mean(predictions == 1):.1%} predicted positive)")


if __name__ == "__main__":
    main()

NameError: name 'NOMINAL_COLUMNS' is not defined